# NC3 — Lansing Findings Correlation Analysis
## Cross-Indicator Relationships Across the Tri-County Region
**Date:** November 04, 2026  
**Prepared for:** Lansing Growth Fund  
**Focus:** Combining employment, resident outcomes, and institutional footprint indicators

This notebook combines key Lansing-area findings from the project's public extracts and examines simple correlations across counties and cities. It is designed as a discussion aid for the community partner meeting — highlighting where institutional presence, manufacturing employment, and resident outcomes move together or diverge.

All numbers are drawn from the committed `Data_Files/` extracts (ACS 2020–2024, QCEW 2024, ACFR, LAUS). Correlations are descriptive only.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:,.1f}'.format)

print('Environment ready — pandas + plotly')

## 1. Combined County-Level Snapshot

Core resident and employment indicators for Clinton, Eaton, and Ingham counties.

In [ ]:
# County-level indicators (ACS 2020-2024 + QCEW 2024 ownership totals)
counties = pd.DataFrame({
    'Geography': ['Clinton County', 'Eaton County', 'Ingham County'],
    'Population': [79626, 109130, 283913],
    'Median HH Income': [88210, 79597, 65526],
    'Poverty Rate (%)': [9.7, 8.3, 16.7],
    'Owner-Occupied (%)': [82.2, 74.2, 55.0],
    'Manufacturing Employment (ACS)': [4506, 8573, 13549],
    'QCEW Total Employment (2024)': [18614, 44501, 155500],
    'QCEW State Gov Employment': [15, 2351, 31363],
    'QCEW NAICS 336 Employment': [341, 4845, 3515]
})
counties['Mfg Share of ACS Workers (%)'] = (counties['Manufacturing Employment (ACS)'] / 
    [39229, 53315, 138634] * 100).round(1)
counties['State Gov Share of QCEW (%)'] = (counties['QCEW State Gov Employment'] / 
    counties['QCEW Total Employment (2024)'] * 100).round(1)

print('Tri-County Combined Snapshot')
display(counties)

## 2. City vs. County Contrast (Lansing & East Lansing)

City-level ACS indicators sit well below the surrounding county medians, especially for income and poverty.

In [ ]:
cities = pd.DataFrame({
    'Geography': ['Lansing city', 'East Lansing city', 'Ingham County', 'MSA'],
    'Median HH Income': [54382, 45687, 65526, 72550],
    'Poverty Rate (%)': [20.1, 33.0, 16.7, 13.5],
    'Population': [113023, 45758, 283913, 472669]
})
print('City vs. Broader Geography')
display(cities)

fig = px.bar(
    cities,
    x='Geography',
    y='Median HH Income',
    title='Median Household Income — City vs. County vs. MSA',
    color='Geography',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(template='plotly_white', height=420, title_x=0.5, showlegend=False)
fig.show()

## 3. Simple Cross-County Correlations

With only three counties the correlation matrix is illustrative rather than statistically robust. It still surfaces the main pattern: higher manufacturing and state-government presence co-occur with lower median income and higher poverty in Ingham.

In [ ]:
# Correlation matrix on the three counties
corr_cols = ['Median HH Income', 'Poverty Rate (%)', 'Owner-Occupied (%)',
             'Mfg Share of ACS Workers (%)', 'State Gov Share of QCEW (%)']
corr = counties[corr_cols].corr().round(2)
print('Cross-County Correlation Matrix (n=3, descriptive only)')
display(corr)

fig = px.imshow(
    corr,
    text_auto=True,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Heatmap — Tri-County Indicators'
)
fig.update_layout(height=480, title_x=0.5)
fig.show()

## 4. Manufacturing & Institutional Presence vs. Resident Outcomes

Scatter views of the three counties. Ingham combines the highest institutional (state government / MSU) and manufacturing presence with the lowest median income and highest poverty rate.

In [ ]:
fig = px.scatter(
    counties,
    x='State Gov Share of QCEW (%)',
    y='Median HH Income',
    size='Population',
    color='Geography',
    text='Geography',
    title='State Government Employment Share vs. Median Household Income',
    labels={'State Gov Share of QCEW (%)': 'State Gov Share of Employment (%)'}
)
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', height=450, title_x=0.5)
fig.show()

fig2 = px.scatter(
    counties,
    x='Mfg Share of ACS Workers (%)',
    y='Poverty Rate (%)',
    size='Population',
    color='Geography',
    text='Geography',
    title='Manufacturing Employment Share vs. Poverty Rate'
)
fig2.update_traces(textposition='top center')
fig2.update_layout(template='plotly_white', height=450, title_x=0.5)
fig2.show()

## 5. GM Footprint in Context

From the ACFR and parcel extracts (see also `NC3_GM_Data_Correlation.ipynb`):

- GM regional employment: 4,549 (2025) — 4th-largest employer, ~2.15% of metro employment
- Combined City of Lansing taxable value: ~$69M (2025), down in rank from #1 in 2016
- Delta Township Assembly core parcel taxable value has risen steadily ($31.5M → $35.0M, 2023–2026)
- Ultium / LG Energy Solution is tracked separately and is not included in GM totals

These institutional numbers sit against the city-level ACS picture (Lansing city median HH income $54k, poverty ~20%) and the county pattern above.

In [ ]:
# Side-by-side comparison for discussion
comparison = pd.DataFrame({
    'Indicator': [
        'GM regional employment (2025)',
        'GM share of metro employment',
        'Lansing city median HH income',
        'Ingham County median HH income',
        'Lansing city poverty rate',
        'Ingham County poverty rate',
        'Delta Twp Assembly TV (2026)'
    ],
    'Value': ['4,549', '2.15%', '$54,382', '$65,526', '20.1%', '16.7%', '$35.0M'],
    'Source': ['ACFR Table 15', 'ACFR Table 15', 'ACS 2020-24', 'ACS 2020-24',
               'ACS 2020-24', 'ACS 2020-24', 'City parcel records']
})
print('Selected Lansing / GM Indicators Side-by-Side')
display(comparison)

## 6. Correlation Takeaways for Partner Discussion

| Pattern | Observation | Implication |
|---------|-------------|-------------|
| Institutional concentration | Ingham has the highest state-government employment share (MSU + state agencies) | Large public institutional footprint does not automatically translate into higher resident median income |
| Manufacturing presence | Eaton has the strongest NAICS 336 / manufacturing concentration (GM Delta Twp Assembly) | Higher manufacturing share coexists with stronger county-level income and lower poverty than Ingham |
| City–county gap | Lansing and East Lansing cities show substantially lower income and higher poverty than their counties | Neighborhood-level mismatch is visible even at the city scale; tract analysis would refine this |
| GM absolute vs. relative | GM employment down ~13% since 2016; taxable value rank has fallen | Absolute footprint remains large, but relative contribution to the metro and city tax base has declined |

**Caveats**
- Three-county correlations are directional only; do not treat coefficients as precise.
- QCEW single-employer cells are frequently suppressed; industry and ownership totals are the reliable public bounds.
- ACS and QCEW measure different concepts (residence-based workers vs. workplace jobs).
- City ACS figures include student populations that affect East Lansing poverty and income statistics.

## 7. Suggested Next Analytical Steps

1. Move from three-county correlations to tract-level analysis (ACS tract file already in `Data_Files/`) to quantify neighborhood mismatch more precisely.
2. Add wage distribution from QCEW for NAICS 336 and state government as a wealth-retention proxy.
3. Overlay LODES origin-destination flows to test whether manufacturing and state-government jobs are captured by local residents.
4. Confirm with the partner which geographic grain (city, county, or tract) is most useful for the first scorecard.
5. Decide whether Ultium / LG Energy Solution should appear as a separate auto-cluster indicator alongside GM.

---
*Notebook prepared for the NC3 community partner meeting. Figures combine project extracts; load the underlying CSVs for production refresh and finer-grain analysis.*